### Notebook 02: Training a Local Model with Databricks Connected Data and MLflow

In this notebook, we train a logistic regression model on the breast cancer data. Instead of loading the dataset locally, we query it directly from Unity Catalog using Databricks Connect. This shows how you can combine scalable data access with fast local training.

We also log everything to MLflow, using a tracking server hosted on Databricks. This includes parameters, metrics, and models, all registered to Unity Catalog for easy access and governance.


In [1]:
# Set up Databricks Connect session
from databricks.connect import DatabricksSession
import os
import config

# Pull connection values from config file
os.environ["DATABRICKS_HOST"] = config.DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = config.DATABRICKS_TOKEN
os.environ["DATABRICKS_CLUSTER_ID"] = config.DATABRICKS_CLUSTER_ID
os.environ["DATABRICKS_WORKSPACE_ID"] = config.DATABRICKS_WORKSPACE_ID

# Initialize Spark session routed through Databricks Connect
spark = DatabricksSession.builder.getOrCreate()


In [2]:
# Core libraries for modeling, evaluation, and logging
import mlflow
from datetime import datetime
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mlflow.models.signature import infer_signature
from mlflow.exceptions import MlflowException


In [3]:
# Define the table and model names to be used in Unity Catalog
CATALOG_NAME = "alexander_booth" 
SCHEMA_NAME = "default"
TABLE_NAME = "breast_cancer_training_data"
MODEL_NAME = "breast_cancer_model"

In [4]:
# Construct table and model URIs
input_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
target_col = "label"
uc_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"

# Timestamped experiment name
today = datetime.now().strftime("%Y%m%d")
experiment_name = f"breast_cancer_{today}"


In [5]:
# Configure MLflow to use Unity Catalog as registry backend
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Define path and artifact location for experiment
experiment_path = f"/Users/alexander.booth@databricks.com/{experiment_name}"

# Log artifacts to a Volume
artifact_location = "dbfs:/Volumes/alexander_booth/default/mlflow_artifacts" 

# Create experiment if it doesn't exist
try:
    mlflow.create_experiment(name=experiment_path, artifact_location=artifact_location)
except MlflowException:
    pass

mlflow.set_experiment(experiment_path)

<Experiment: artifact_location='dbfs:/Volumes/alexander_booth/default/mlflow_artifacts', creation_time=1747420714817, experiment_id='913238801103218', last_update_time=1747420714817, lifecycle_stage='active', name='/Users/alexander.booth@databricks.com/breast_cancer_20250516', tags={'mlflow.experiment.sourceName': '/Users/alexander.booth@databricks.com/breast_cancer_20250516',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'alexander.booth@databricks.com',
 'mlflow.ownerId': '8950113264559567'}>

In [6]:
# Load Delta table from Unity Catalog as Spark DataFrame, then convert to pandas
df = spark.read.table(input_table)
df_pd = df.toPandas()
print(df_pd.shape)

# Split into features and label, then train/test split
X = df_pd.drop(columns=["label"])
y = df_pd["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Preview training data
X_train.head(5)

(569, 31)


,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension
287,12.89,13.12,81.89,515.9,0.06955,0.03729,0.02260,0.01171,0.1337,0.05581,...,13.62,15.54,87.40,577.0,0.09616,0.1147,0.1186,0.05366,0.2309,0.06915
512,13.40,20.52,88.64,556.7,0.11060,0.14690,0.14450,0.08172,0.2116,0.07325,...,16.41,29.66,113.30,844.4,0.15740,0.3856,0.5106,0.20510,0.3585,0.11090
402,12.96,18.29,84.18,525.2,0.07351,0.07899,0.04057,0.01883,0.1874,0.05899,...,14.13,24.61,96.31,621.9,0.09329,0.2318,0.1604,0.06608,0.3207,0.07247
446,17.75,28.03,117.30,981.6,0.09997,0.13140,0.16980,0.08293,0.1713,0.05916,...,21.53,38.54,145.40,1437.0,0.14010,0.3762,0.6399,0.19700,0.2972,0.09075
210,20.58,22.14,134.70,1290.0,0.09090,0.13480,0.16400,0.09561,0.1765,0.05024,...,23.24,27.84,158.30,1656.0,0.11780,0.2920,0.3861,0.19200,0.2909,0.05865


In [7]:
# Define ML pipeline: scale features, then fit logistic regression
pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

# Use current time for unique run name
now = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"logistic-regression-{now}"

# Basic hyperparameter grid for tuning
param_grid = {
    "lr__C": [0.01, 0.1, 1.0], # Regularization strength
    "lr__penalty": ["l2"] # L2 penalty
}

# 5-fold grid search for demo purposes
grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5, verbose=2)

# Track run with MLflow
with mlflow.start_run(run_name=run_name) as run:
    # grid.fit does both training and validation
    grid.fit(X_train, y_train)

    # Evaluate on test set
    y_pred = grid.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", acc)

    # Log parameters (explicitly, in case autolog isn't used)
    for param_name, value in grid.best_params_.items():
        mlflow.log_param(param_name, value)

    # Log model with signature for schema inference
    signature = infer_signature(X_train, y_pred)
    mlflow.sklearn.log_model(
        sk_model=grid.best_estimator_,
        artifact_path="model",
        signature=signature
    )

    # Register the model to Unity Catalog for governance and sharing
    model_uri = f"runs:/{run.info.run_id}/model"
    mlflow.register_model(
        model_uri=model_uri,
        name=uc_model_name
    )

Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV] END .........................lr__C=0.01, lr__penalty=l2; total time=   0.0s
[CV] END .........................lr__C=0.01, lr__penalty=l2; total time=   0.0s
[CV] END .........................lr__C=0.01, lr__penalty=l2; total time=   0.0s
[CV] END .........................lr__C=0.01, lr__penalty=l2; total time=   0.0s
[CV] END .........................lr__C=0.01, lr__penalty=l2; total time=   0.0s
[CV] END ..........................lr__C=0.1, lr__penalty=l2; total time=   0.0s
[CV] END ..........................lr__C=0.1, lr__penalty=l2; total time=   0.0s
[CV] END ..........................lr__C=0.1, lr__penalty=l2; total time=   0.0s
[CV] END ..........................lr__C=0.1, lr__penalty=l2; total time=   0.0s
[CV] END ..........................lr__C=0.1, lr__penalty=l2; total time=   0.0s
[CV] END ..........................lr__C=1.0, lr__penalty=l2; total time=   0.0s
[CV] END ..........................lr__C=1.0, lr_

Registered model 'alexander_booth.default.breast_cancer_model' already exists. Creating a new version of this model...
Created version '3' of model 'alexander_booth.default.breast_cancer_model'.


🏃 View run logistic-regression-20250516_133840 at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/913238801103218/runs/c15f64a0bbb34d9c8e65e5559e439fa1
🧪 View experiment at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/913238801103218


---

### ✅ Wrapping Up: Local Modeling, Logged Remotely

In this notebook, we trained a logistic regression model locally using scikit-learn, while pulling data from Unity Catalog via Databricks Connect. Our entire experiment was tracked in MLflow, using a remote tracking server on Databricks and storing artifacts in Unity Catalog.

This setup is great for fast iteration with familiar Python tools, without sacrificing observability or governance. In the next notebook, we’ll explore what happens when we try to push training to the Databricks cluster itself using PySpark ML.

Let's see what happens...
